In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# Define the path to the Dataset folder
folder_path = '/content/drive/MyDrive/Sanskrit_LLM/Dataset'

# List contents of the folder to identify the file
print(f"Contents of {folder_path}:")
if os.path.exists(folder_path):
    for item in os.listdir(folder_path):
        print(item)
else:
    print(f"Folder not found: {folder_path}")

Contents of /content/drive/MyDrive/Sanskrit_LLM/Dataset:
disease_ontology.json
Raw_texts
Book_Names.json
NATIONAL AYURVEDA MORBIDITY CODES.xlsx
Sample_KG_Data
charaka_dataset.json
charaka_tree_dataset.json
charaka_tree_cleaned.json
Top Diseases.gsheet
DATASET.gsheet


In [3]:
import pandas as pd

# The previous attempt failed because the file was identified as 'DATASET.gsheet' not 'DATASET.csv'.
file_name = 'DATASET.gsheet' # Correcting the filename based on the directory listing.
file_path = os.path.join(folder_path, file_name)

try:
    # Reading a Google Sheet directly from Google Drive usually requires specific libraries
    # like 'gspread' and proper authentication, or exporting it to a CSV/Excel format.
    # pd.read_csv will not work directly with a .gsheet file.
    print(f"The file '{file_name}' is a Google Sheet shortcut. It cannot be read directly with `pandas.read_csv`.")
    print("To read a Google Sheet, you typically need to use the `gspread` library and authorize access, or export the sheet to CSV/Excel format and then read it.")
    print("I will guide you through reading the Google Sheet using `gspread` in the next step.")

except Exception as e:
    print(f"An unexpected error occurred: {e}")

The file 'DATASET.gsheet' is a Google Sheet shortcut. It cannot be read directly with `pandas.read_csv`.
To read a Google Sheet, you typically need to use the `gspread` library and authorize access, or export the sheet to CSV/Excel format and then read it.
I will guide you through reading the Google Sheet using `gspread` in the next step.


In [4]:
# Install gspread if not already installed
!pip install gspread oauth2client

import gspread
from google.colab import auth
import pandas as pd
import google.auth # Import google.auth

# Authenticate Google Colab to access Google Drive and Sheets
# This will prompt you to authorize access in your browser.
auth.authenticate_user()

# Get default credentials and authorize gspread with them
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

try:
    # Open the spreadsheet by its name. Assuming the actual Google Sheet is also named 'DATASET'.
    # If the sheet has a different name, you will need to update this.
    spreadsheet_name = 'DATASET'
    worksheet = gc.open(spreadsheet_name).sheet1 # Opens the first worksheet

    # Get all values from the worksheet as a list of lists
    data = worksheet.get_all_values()

    # Convert to pandas DataFrame
    # The first row is assumed to be the header
    df_gsheet = pd.DataFrame(data[1:], columns=data[0])

    print(f"Successfully read Google Sheet '{spreadsheet_name}' into a DataFrame.")
    display(df_gsheet.head())

except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Google Sheet '{spreadsheet_name}' not found. Please ensure the name is correct and you have access to it.")
except Exception as e:
    print(f"An error occurred while reading the Google Sheet: {e}")

Successfully read Google Sheet 'DATASET' into a DataFrame.


,3,NAMC_ID,NAMC_CODE,NAMC_term,NAMC_term_diacritical,NAMC_term_DEVANAGARI,Short_definition,Long_definition,Ontology_branches,Name English,Name English Under Index,Primary Index Related
0,1,1,AYU,vyAdhi-viniScayaH,vyādhi-viniścayaḥ,व्याधि-विनिश्चयः,-,-,-,diagnostic conditions,-,-
1,2,2,DIS,vikAraH,vikāraḥ,विकारः,-,-,-,disease/disorder,-,-
2,3,3,A,doShavaiShamyam,dōṣavaiṣamyam,दोषवैषम्यम्,-,-,-,derangement of dōṣa,-,-
3,4,4,AA,vAtavyAdhiH,vātavyādhiḥ,वातव्याधिः,-,-,-,disorders due to vāta,-,-
4,5,5,AAA,doShAvasthA (vAta),dōṣāvasthā (vāta),दोषावस्था (वात),-,-,-,morbid state of dōṣa,-,-


In [7]:
import re
import pandas as pd

# Copy dataframe
initial_df = df_gsheet.copy()

col = "Long_definition"

# -------------------------------
# STEP 1: Filter rows starting with phrase
# -------------------------------
start_phrase = "the disorder is characterized by"

filtered_df = initial_df[
    initial_df[col].str.lower().str.startswith(start_phrase, na=False)
].copy()

print("Filtered rows:", len(filtered_df))

# -------------------------------
# STEP 2: Remove the starting phrase
# -------------------------------
filtered_df[col] = filtered_df[col].str.replace(
    start_phrase, "", regex=False, case=False
).str.strip()

# -------------------------------
# STEP 3: Extract IAST + English pairs
# format: term [meaning], term [meaning]
# -------------------------------
def extract_pairs(text):
    if pd.isna(text):
        return []

    text = str(text)

    # Unicode-safe capture (fixes truncated words issue)
    matches = re.findall(
        r'([^\s\[]+)\s*\[([^\]]+)\]',
        text
    )

    return matches

filtered_df["pairs"] = filtered_df[col].apply(extract_pairs)

# -------------------------------
# STEP 4: Split terms like a/b
# -------------------------------
all_pairs = []

for pair_list in filtered_df["pairs"]:
    for term, meaning in pair_list:

        terms = term.split('/')

        for t in terms:
            t = t.strip()
            meaning = meaning.strip()

            # skip bad terms
            if "?" in t:
                continue
            if len(t) < 2:
                continue

            all_pairs.append((t, meaning))

# -------------------------------
# STEP 5: Create DataFrame
# -------------------------------
pairs_df = pd.DataFrame(all_pairs, columns=["IAST_term", "English_meaning"])

# -------------------------------
# STEP 6: Clean
# -------------------------------
pairs_df = pairs_df.drop_duplicates()
pairs_df = pairs_df.dropna()

pairs_df["IAST_term"] = pairs_df["IAST_term"].str.strip()
pairs_df["English_meaning"] = pairs_df["English_meaning"].str.strip()

# -------------------------------
# STEP 7: Training format
# -------------------------------
pairs_df["input_text"] = "symptom: " + pairs_df["English_meaning"]
pairs_df["target_text"] = pairs_df["IAST_term"]

# -------------------------------
# STEP 8: Save to Google Sheet
# -------------------------------
data_to_write = [pairs_df.columns.values.tolist()] + pairs_df.values.tolist()

try:
    spreadsheet = gc.open(spreadsheet_name)
    worksheet_title = "IAST_Cleaned_Output"

    try:
        output_worksheet = spreadsheet.worksheet(worksheet_title)
        output_worksheet.clear()
    except:
        output_worksheet = spreadsheet.add_worksheet(
            title=worksheet_title,
            rows=len(data_to_write),
            cols=len(pairs_df.columns)
        )

    output_worksheet.update(data_to_write, 'A1')

    print(f"✅ Saved to sheet: {worksheet_title}")

except Exception as e:
    print(f"❌ Error: {e}")

# -------------------------------
# FINAL OUTPUT
# -------------------------------
print("✅ Total clean pairs:", len(pairs_df))
display(pairs_df.head(20))

Filtered rows: 1177
✅ Saved to sheet: IAST_Cleaned_Output
✅ Total clean pairs: 6222


,IAST_term,English_meaning,input_text,target_text
0,hikkā,hiccup,symptom: hiccup,hikkā
1,śvāsa,breathlessness/difficult breathing,symptom: breathlessness/difficult breathing,śvāsa
2,cakṣurādīnāmupaghāta,impairment of sense organs viz. eye,symptom: impairment of sense organs viz. eye,cakṣurādīnāmupaghāta
3,pīnasa,"cold, catarrh","symptom: cold, catarrh",pīnasa
4,ardita,facial paralysis,symptom: facial paralysis,ardita
5,tr̥ṭ,thirst,symptom: thirst,tr̥ṭ
6,kāsaḥ,cough,symptom: cough,kāsaḥ
7,kaṇṭharōdha,choking of throat/stopping or lowering voice,symptom: choking of throat/stopping or lowerin...,kaṇṭharōdha
8,manōbhraṁśa,mental disturbance,symptom: mental disturbance,manōbhraṁśa
9,chardiḥ,vomiting,symptom: vomiting,chardiḥ
